# local_constant

> 对应代码：`EconometricsCode/code_in_notes/Chap.3/local_constant.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.3`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.3")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


清理环境并设置随机种子，保证模拟结果可复现。

In [2]:
%%stata
clear
set more off
// 生成数据
set seed 19880505


. clear

. set more off

. // 生成数据
. set seed 19880505

. 


我们生成 300 个观测：解释变量 x 服从 [0,2] 上的均匀分布，被解释变量 y 由非线性函数 exp(sin(x^3)) 加上标准正态噪声构成。这个真实函数具有明显的非线性特征，是检验非参数回归方法的好例子。

In [3]:
%%stata
set obs 300
gen x=2*runiform()
gen y=exp(sin(x^3))+rnormal()


. set obs 300
Number of observations (_N) was 0, now 300.

. gen x=2*runiform()

. gen y=exp(sin(x^3))+rnormal()

. 


我们演示局部常数（Nadaraya-Watson）回归在 x=2 处的估计。首先用正态密度函数 normalden 构造核权重 w，带宽取 0.1：距离 x=2 越近的观测权重越大，越远的观测权重按正态密度迅速衰减。随后用重要性权重 [iw=w] 做只含常数项的加权回归，这等价于对 x=2 附近的 y 做加权平均——正是局部常数估计量的定义。

In [4]:
%%stata
// 局部常数回归
gen w=normalden((x-2)/0.1)
reg y [iw=w]


. // 局部常数回归
. gen w=normalden((x-2)/0.1)

. reg y [iw=w]

      Source |       SS           df       MS      Number of obs   =         8
-------------+----------------------------------   F(0, 7)         =      0.00
       Model |           0         0           .   Prob > F        =         .
    Residual |  10.3051056         7  1.47215794   R-squared       =    0.0000
-------------+----------------------------------   Adj R-squared   =    0.0229
       Total |  10.3051056         7  1.47215794   Root MSE        =    1.1994

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
       _cons |   2.069896   .4197584     4.93   0.002     1.077326    3.062467
------------------------------------------------------------------------------

. 


截距项 _b[_cons] 就是 m(2)=E(y|x=2) 的局部常数估计，我们将其打印出来查看。

In [5]:
%%stata
di _b[_cons]

2.0698965
